# Q1 Instantaneous-Basis Sink-Truth Validation

This notebook compares the fixed-basis compact interpolated model against a dense instantaneous-basis compact reference with gauge Hamiltonian and field-dependent sink decay. It also includes patch-grid and gauge ablations.

In [ ]:
from __future__ import annotations

import importlib.util
import pathlib
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"font.size": 14})

script_path = pathlib.Path("validate_q1_instantaneous_sink_truth.py")
if not script_path.exists():
    script_path = pathlib.Path("examples/effective hamiltonian/validate_q1_instantaneous_sink_truth.py")
spec = importlib.util.spec_from_file_location("validate_q1_instantaneous_sink_truth", script_path)
v = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = v
spec.loader.exec_module(v)

ehr = v.load_runtime_module()

## Configuration

`REFERENCE_POINTS` controls the dense instantaneous-basis reference grid. `CANDIDATE_GRIDS` controls the fixed-basis model patch grid used for the candidate.

In [ ]:
REFERENCE_POINTS = 21
CANDIDATE_GRIDS = ["production", "uniform_2p5", "uniform_1p25"]
RUN_GAUGE_ABLATION = True

cases = v.default_cases()
[(name, field_at(0.0), field_at(v.T_FINAL)) for name, field_at, _ in cases]

## Reference Solve

The instantaneous reference is built and propagated once, then reused for all fixed-basis candidate grids.

In [ ]:
reference_fields_vcm = np.linspace(0.0, 50.0, REFERENCE_POINTS)

start = time.perf_counter()
reference_model = v.build_instantaneous_model(ehr, reference_fields_vcm)
reference_data = v.precompute_instantaneous_reference(ehr, reference_model)
reference_setup_s = time.perf_counter() - start

rho0_reference = v.make_initial_state(reference_data)
solved_reference_cases = {}
for name, field_at, field_dot_at in cases:
    case_start = time.perf_counter()
    solved_reference_cases[name] = v.solve_case_data(
        name,
        field_at,
        field_dot_at,
        reference_data,
        rho0_reference,
    )
    print(f"reference {name}: {time.perf_counter() - case_start:.2f} s")

print(f"reference setup: {reference_setup_s:.2f} s")
print(f"reference states: {reference_data['n_states']}")
print(f"reference sinks: {reference_data['sink_labels']}")

## Candidate Grid Sweep

Each candidate grid is built separately and compared against the same solved instantaneous reference.

In [ ]:
candidate_outputs = []

for grid_name in CANDIDATE_GRIDS:
    fields = v.candidate_grid(grid_name)
    start = time.perf_counter()
    candidate_model = v.build_candidate_model(ehr, fields)
    candidate_data = v.precompute_candidate(ehr, candidate_model)
    candidate_setup_s = time.perf_counter() - start

    results = v.run_candidate_vs_reference(
        candidate_data,
        reference_data,
        solved_reference_cases=solved_reference_cases,
    )
    candidate_outputs.append(
        {
            "grid": grid_name,
            "setup_s": candidate_setup_s,
            "fields_vcm": fields,
            "n_states": candidate_data["n_states"],
            "sink_labels": candidate_data["sink_labels"],
            "results": results,
        }
    )
    print(f"{grid_name}: setup {candidate_setup_s:.2f} s, fields={fields.size}")

## Summary Table

In [ ]:
summary_rows = []
for output in candidate_outputs:
    for result in output["results"]:
        row = {
            "grid": output["grid"],
            "n_fields": len(output["fields_vcm"]),
            "case": result["case"],
            "setup_s": output["setup_s"],
            "candidate_elapsed_s": result["solver"]["candidate_elapsed_s"],
            "reference_elapsed_s": result["solver"]["reference_elapsed_s"],
            "candidate_nfev": result["solver"]["candidate_nfev"],
            "reference_nfev": result["solver"]["reference_nfev"],
            "candidate_photons": result["candidate"]["photons"],
            "reference_photons": result["reference"]["photons"],
            "candidate_final_sink": result["candidate"]["final_total_sink_population"],
            "reference_final_sink": result["reference"]["final_total_sink_population"],
        }
        row.update(result["errors"])
        summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_df

## Sink Populations

In [ ]:
sink_rows = []
for output in candidate_outputs:
    for result in output["results"]:
        for label, candidate, reference in zip(
            output["sink_labels"],
            result["candidate"]["final_sink_populations"],
            result["reference"]["final_sink_populations"],
            strict=True,
        ):
            sink_rows.append(
                {
                    "grid": output["grid"],
                    "case": result["case"],
                    "sink": label,
                    "candidate": candidate,
                    "reference": reference,
                    "abs_error": candidate - reference,
                }
            )

sink_df = pd.DataFrame(sink_rows)
sink_df

## Gauge Ablation

This compares the instantaneous reference with and without the moving-basis gauge Hamiltonian. Large differences here mean the gauge term is not optional for time-dependent fields.

In [ ]:
gauge_df = pd.DataFrame()
if RUN_GAUGE_ABLATION:
    gauge_results = v.compare_references(
        "with_gauge",
        reference_data,
        "without_gauge",
        v.without_gauge(reference_data),
    )
    gauge_rows = []
    for result in gauge_results:
        row = {
            "case": result["case"],
            "lhs_reference": result["lhs_reference"],
            "rhs_reference": result["rhs_reference"],
            "lhs_photons": result["candidate"]["photons"],
            "rhs_photons": result["reference"]["photons"],
        }
        row.update(result["errors"])
        gauge_rows.append(row)
    gauge_df = pd.DataFrame(gauge_rows)

gauge_df

## Error Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), constrained_layout=True)
metrics = ["photons_rel", "excited_integral_rel", "max_excited_rel"]
titles = ["Photon integral", "Excited integral", "Max excited trace"]

for ax, metric, title in zip(axes, metrics, titles, strict=True):
    for case, sub in summary_df.groupby("case", sort=False):
        ax.plot(sub["grid"], np.abs(sub[metric]), marker="o", label=case)
    ax.set_yscale("log")
    ax.set_title(title)
    ax.set_ylabel(f"abs({metric})")
    ax.tick_params(axis="x", rotation=25)
    ax.grid(True, which="both", alpha=0.3)

axes[0].legend()
plt.show()

## Interpretation

Use the integrated photon and excited-population errors as the primary accuracy metrics. Final excited population can have a much larger relative error when the final excited population is small. If densifying the candidate patch grid improves the error, the production grid is the limiting factor. If the error does not converge cleanly with the candidate grid, the difference is likely from fixed-basis versus instantaneous-basis dynamics or from reference-grid interpolation.